## Creación de dbf1 con chroma y llm

## **Ojo:** esta es la copia del código usado para crear dbf1 y actualizarla de forma demasiado artesanal.
La carga de los embeddings a db_f1 toma más de 6 hrs. 

### **CUALQUIER ADICIÓN TIENE QUE HACERSE CON UPSERT**

In [1]:
from dotenv import load_dotenv
import os

from IPython.display import HTML, Markdown, display
import pandas as pd
import openai

from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

import tiktoken
import tqdm
import re
import ast


# Add this line to load variables from .env file
load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')  # Retrieve the API key
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')  # Retrieve the API key
LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')  # Retrieve the API key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


client = OpenAI()

client.embeddings.create(
  model='text-embedding-3-large', #"text-embedding-ada-002",
  input="The food was delicious and the waiter...",
  encoding_format="float"
)

embedding_fun_openai = OpenAIEmbeddingFunction(api_key=os.environ.get('OPENAI_API_KEY'), model_name="text-embedding-ada-002")

In [2]:
! git branch

* db_f1
  master
  test_enc_1
  test_enc_2


In [3]:
# carga de pickle con datos de encuestas
import pickle

ruta_enc= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas'

with open(f'{ruta_enc}/los_mex_dict.pkl', 'rb') as f:
    los_mex_dict = pickle.load(f)
    print('los_mex_dict cargado --  leer readme_dict para info')


enc_dict = los_mex_dict['enc_dict']
enc_nom_dict = los_mex_dict['enc_nom_dict']
pregs_dict = los_mex_dict['pregs_dict']
ses_dict = los_mex_dict['ses_dict']

mkdown_tables = los_mex_dict['mkdown_tables']
df_tables = los_mex_dict['df_tables']

los_mex_dict cargado --  leer readme_dict para info


### funciones

In [4]:
# funciones para calcular dfs y tablas en markdown

def dataframe_to_markdown(df):
    """
    Converts a pandas DataFrame to a markdown table string with index name and values.
    
    Parameters:
        df (pd.DataFrame): The DataFrame to convert.
        
    Returns:
        str: A markdown table string representation of the DataFrame including index.
    """
    # Get column headers
    headers = df.columns.tolist()
    
    # Get index name (use empty string if None)
    index_name = df.index.name if df.index.name is not None else ""
    
    # Create header row with index name
    header_row = "| " + str(index_name) + " | " + " | ".join(str(col) for col in headers) + " |"
    
    # Create separator row (defines alignment)
    separator_row = "| --- | " + " | ".join(["---" for _ in headers]) + " |"
    
    # Create data rows with index values
    data_rows = []
    for idx, row in df.iterrows():
        formatted_values = []
        for val in row:
            if isinstance(val, float):
                formatted_values.append(f"{val:.2f}")
            else:
                formatted_values.append(str(val))
        
        # Include the index value as first column
        data_rows.append("| " + str(idx) + " | " + " | ".join(formatted_values) + " |")
    
    # Combine all parts
    markdown_table = "\n".join([header_row, separator_row] + data_rows)
    
    return markdown_table


def calculate_weighted_proportion(df, categorical_var, weight_var='ponderador', normalize=True):
    """
    Calculate the weighted proportion of a categorical variable.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The DataFrame containing the data
    categorical_var : str
        The name of the categorical variable column
    weight_var : str, default 'ponderador'
        The name of the weight variable column
    normalize : bool, default True
        If True, normalize weights to sum to 1
        
    Returns:
    --------
    pandas.DataFrame
        A DataFrame with categories as index and weighted proportions as values
    """
    # Ensure the weight variable is numeric
    df = df.copy()
    df[weight_var] = df[weight_var].astype(float)
    
    # Fill missing weights with 0
    df[weight_var] = df[weight_var].fillna(0)
    
    # Normalize weights if requested
    if normalize:
        df[weight_var] = df[weight_var] / df[weight_var].sum()
    
    # Group by the categorical variable and sum the weights
    weighted_counts = df.groupby(categorical_var)[weight_var].sum()
    
    # Create a DataFrame with the results
    result = weighted_counts.to_frame(name='proportion')
    
    return result


In [5]:
# funcinoes para llamar llm, contar tokens y batch documentos


def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18'):
    """Get a simple answer from OpenAI models without chatbot functionality."""
    from openai import OpenAI
    
    client = OpenAI(
        api_key=os.environ.get("OPENAI_API_KEY"),
    )
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model=model, #'gpt-4.1-nano-2025-04-14', #"gpt-4o",
        messages=messages
    )
    
    return response.choices[0].message.content


def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Update the token count calculation in batch_documents to use num_tokens_from_string
# 8192 is the token limit for the model
def batch_documents(docs, ids, max_tokens=8192, encoding_name="cl100k_base"):
    """
    Batches documents and their corresponding IDs while respecting the token limit.

    Parameters:
        docs: List of documents to batch.
        ids: List of IDs corresponding to the documents.
        max_tokens: Maximum token limit for each batch.
        encoding_name: Encoding name for token calculation.

    Returns:
        List of batches, where each batch is a tuple of (batched_docs, batched_ids).
    """
    batches = []
    current_batch_docs = []
    current_batch_ids = []
    current_token_count = 0

    for doc, doc_id in zip(docs, ids):
        doc_token_count = num_tokens_from_string(doc, encoding_name)  # Use the token counting function
        
        print(f"Document ID: {doc_id}, Token Count: {doc_token_count}")
        
        if current_token_count + doc_token_count > max_tokens:
            # Save the current batch and start a new one
            batches.append((current_batch_docs, current_batch_ids))
            current_batch_docs = []
            current_batch_ids = []
            current_token_count = 0

        # Add the document to the current batch
        current_batch_docs.append(doc)
        current_batch_ids.append(doc_id)
        current_token_count += doc_token_count

    # Add the last batch if it has any documents
    if current_batch_docs:
        batches.append((current_batch_docs, current_batch_ids))

    return batches


## Pipeline: Structured Summaries with LangChain + Pydantic
Define a Pydantic model, output parser, and PromptTemplate for `create_prompt_sum2`, along with a helper to call the LLM and parse structured results.

In [ ]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser
from langchain import PromptTemplate, LLMChain
from langchain.chat_models import ChatOpenAI
import pickle
from tqdm import tqdm

# 1. Define Pydantic model for an entry
class SummaryEntry(BaseModel):
    # Pydantic is used here to define a structured schema for the expected output.
    # This ensures that the output from the LLM adheres to a specific format,
    # making it easier to validate and process the results.
    summary: str
    implication: str

# 2. Create parser for single entries
parser_summary = PydanticOutputParser(pydantic_object=SummaryEntry)
format_instructions = parser_summary.get_format_instructions()
 # LangChain's PydanticOutputParser generates instructions for the LLM to format its output
 # according to the Pydantic model. This ensures compatibility between the LLM's response
 # and the expected schema.

# 3. Helper to call LLM and parse each table into a dict


def create_prompt_sum2(
    table_key: str,
    mk_tables: dict[str, str],
    nsnc_threshold: float = 2.0,
    topic: str | None = None,
    format_instructions: str = ""
) -> str:
    # This function generates a prompt for the LLM to analyze a table.
    # It includes specific tasks and format instructions to guide the LLM's response.
    md = mk_tables[table_key]
    tpc = topic or md.split('|')[1].replace('_',' ').lower().strip()
    
    sub1 = (
      f"Write a concise summary of the table. "
      f"If the “no answer/does not know” row exceeds {nsnc_threshold:.1f}%, "
      f"mention it and explain its significance."
    )
    sub2 = (
      f"Then answer: “Why is this table important for an expert in '{tpc}'?” "
      f"and how others can leverage these insights."
    )
    
    return (
      f"You are a bilingual English/Spanish data analyst expert in {tpc}'.\n"
      f"Analyze this TABLE:\n\n{md}\n\n"
      "Tasks:\n"
      f"A) {sub1}\n"
      f"B) {sub2}\n\n"
      "Respond with **valid JSON** matching:\n"
      "{\n  \"summary\": string,\n  \"implication\": string\n}"
      f"\n\n{format_instructions}"
    )

def get_answer(prompt, system_prompt=None, model='gpt-4o-mini-2024-07-18', temperature=0.9):
    # This function sends the prompt to the LLM and retrieves the response.
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user",   "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

def get_structured_summary(table_key: str, model_name: str = 'gpt-4o-mini-2024-07-18', temperature: float = 0.9):
    # This function combines the prompt creation and LLM call,
    # then parses the response using the PydanticOutputParser.
    prompt = create_prompt_sum2(table_key, mkdown_tables) + "\n\n" + format_instructions
    content = get_answer(prompt, model=model_name, temperature=temperature)
    parsed = parser_summary.parse(content)
    return parsed.model_dump()  # Returns the parsed response as a dictionary.

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    # Utility function to calculate the number of tokens in a string.
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Update the token count calculation in batch_documents to use num_tokens_from_string
# 8192 is the token limit for the model
def batch_documents(docs, ids, max_tokens=8192, encoding_name="cl100k_base"):
    # This function batches documents and their IDs while respecting the token limit.
    # It ensures that each batch does not exceed the maximum token limit,
    # which is crucial for efficient processing with the LLM.
    batches = []
    current_batch_docs = []
    current_batch_ids = []
    current_token_count = 0

    for doc, doc_id in zip(docs, ids):
        doc_token_count = num_tokens_from_string(doc, encoding_name)  # Use the token counting function
        
        # print(f"Document ID: {doc_id}, Token Count: {doc_token_count}")
        
        if current_token_count + doc_token_count > max_tokens:
            # Save the current batch and start a new one
            batches.append((current_batch_docs, current_batch_ids))
            current_batch_docs = []
            current_batch_ids = []
            current_token_count = 0

        # Add the document to the current batch
        current_batch_docs.append(doc)
        current_batch_ids.append(doc_id)
        current_token_count += doc_token_count

    # Add the last batch if it has any documents
    if current_batch_docs:
        batches.append((current_batch_docs, current_batch_ids))

    return batches


def batch_process_tables(mkdown_tables, batch_size=8192):
    # This function processes markdown tables in batches,
    # saving checkpoints after each batch to ensure progress is not lost.
    prompts = [create_prompt_sum2(key, mkdown_tables) for key in mkdown_tables.keys()]
    keys = list(mkdown_tables.keys())
    
    batches = batch_documents(prompts, keys, max_tokens=batch_size, encoding_name="cl100k_base")
    
    # Initialize results dictionary
    structured_results = {}
    
    # Process each batch
    with tqdm(total=len(keys), desc="Processing Batches") as pbar:
        for batch_docs, batch_keys in batches:
            for prompt, key in zip(batch_docs, batch_keys):
                try:
                    structured_results[key] = get_structured_summary(key)
                except Exception as e:
                    structured_results[key] = {'error': str(e)}
                pbar.update(1)
            
            # Save checkpoint after each batch
            with open('structured_summary_checkpoint.pkl', 'wb') as f:
                pickle.dump(structured_results, f)
            print(f"Checkpoint saved after processing batch with keys: {batch_keys}")
    
    return structured_results



structured_results = batch_process_tables(mkdown_tables)

In [51]:
los_mex_dict['llm_results'] = structured_results
with open(f'{ruta_enc}/los_mex_dict.pkl', 'wb') as f:
    pickle.dump(los_mex_dict, f)
    print('los_mex_dict actualizado --  leer readme_dict para info')

los_mex_dict.keys()

los_mex_dict actualizado --  leer readme_dict para info


dict_keys(['enc_dict', 'enc_nom_dict', 'pregs_dict', 'ses_dict', 'readme_dict', 'mkdown_tables', 'df_tables', 'llm_results'])

## Embed Structured Summaries into ChromaDB (DuckDB+Parquet)
Convert the full `structured_results` dict into a single document and embed it for retrieval.

In [49]:
from tqdm import tqdm
import chromadb
from chromadb import PersistentClient                            
from chromadb.config import Settings, DEFAULT_TENANT, DEFAULT_DATABASE  
import tiktoken

answers_dict = structured_results

# --------------------------------------------------------------------
# 1) Configure your persistent client (v0.4+)
# --------------------------------------------------------------------
DB_PERSIST_DIR = "./db_f1"
DB_NAME        = "enc_dbf1"
reset_db       = False

client = PersistentClient(
    path     = DB_PERSIST_DIR,
    settings = Settings(allow_reset=True),   # allow resetting on-disk state
    tenant   = DEFAULT_TENANT,
    database = DEFAULT_DATABASE,
)  # :contentReference[oaicite:0]{index=0}

# --------------------------------------------------------------------
# 2) Create or reset the collection
# --------------------------------------------------------------------
if reset_db:
    # deletes the whole collection directory if it exists
    try:
        client.delete_collection(name=DB_NAME)     # :contentReference[oaicite:1]{index=1}
    except ValueError:
        pass

db_f1 = client.get_or_create_collection(
    name               = DB_NAME,
    embedding_function = embedding_fun_openai
)

# --------------------------------------------------------------------
# 3) Build your 3-facet docs (question, summary, implications)
# --------------------------------------------------------------------
docs, ids, metadatas = [], [], []
for qid in pregs_dict:
    q_text = str(pregs_dict[qid]).strip()

    if answers_dict.get(qid, "") == "incorrect" or answers_dict.get(qid, "") == "":
        print(f"Skipping incorrect answer for {qid}")
        continue

    s_text = str(answers_dict.get(qid, "")['summary']).strip()
    i_text = str(answers_dict.get(qid, "")['implication']).strip()

    for facet, text in [
        ("question",     q_text),
        ("summary",      s_text),
        ("implications", i_text),
    ]:
        if text:
            docs.append(text)
            ids.append(f"{qid}__{facet}")
            metadatas.append({"qid": qid, "type": facet})

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(string))

def batch_documents(docs, ids, metadatas,
                    max_tokens=8192, encoding_name="cl100k_base"):
    batches = []
    cur_docs, cur_ids, cur_metas, cur_tokens = [], [], [], 0

    for doc, did, meta in zip(docs, ids, metadatas):
        ntoks = num_tokens_from_string(doc, encoding_name)
        if cur_tokens + ntoks > max_tokens:
            batches.append((cur_docs, cur_ids, cur_metas))
            cur_docs, cur_ids, cur_metas, cur_tokens = [], [], [], 0

        cur_docs.append(doc)
        cur_ids.append(did)
        cur_metas.append(meta)
        cur_tokens += ntoks

    if cur_docs:
        batches.append((cur_docs, cur_ids, cur_metas))
    return batches

# --------------------------------------------------------------------
# 4) Batch, embed (with inner progress bar), and add to db_f1
# --------------------------------------------------------------------
batches = batch_documents(
    docs, ids, metadatas,
    max_tokens=8192,
    encoding_name="cl100k_base"
)

for batch_docs, batch_ids, batch_metas in tqdm(batches, desc="Adding batches to DB"):
    embeddings = []
    for doc in tqdm(batch_docs, desc="  Embedding docs", leave=False):
        emb = embedding_fun_openai([doc])[0]              
        embeddings.append(emb)

    db_f1.add(
        documents = batch_docs,
        ids       = batch_ids,
        metadatas = batch_metas,
        embeddings= embeddings
    )


print("Total vectors:", db_f1.count())
print("Sample peek:", db_f1.peek())


Skipping incorrect answer for p1_1|IDE
Skipping incorrect answer for p2_1|IDE
Skipping incorrect answer for p26_1|IDE
Skipping incorrect answer for p61_1_o|IDE
Skipping incorrect answer for p62_1_o|IDE
Skipping incorrect answer for p63_1_o|IDE
Skipping incorrect answer for p35_1|MED
Skipping incorrect answer for p6_1|CUL
Skipping incorrect answer for p26_1|SAL
Skipping incorrect answer for p26_2|SAL
Skipping incorrect answer for p26_3|SAL
Skipping incorrect answer for p26_4|SAL
Skipping incorrect answer for p26_5|SAL
Skipping incorrect answer for p26_6|SAL
Skipping incorrect answer for p26_7|SAL
Skipping incorrect answer for p26_8|SAL
Skipping incorrect answer for p26_9|SAL
Skipping incorrect answer for p26_10|SAL
Skipping incorrect answer for p26_11|SAL
Skipping incorrect answer for p1_1|SOC
Skipping incorrect answer for p32h|SOC
Skipping incorrect answer for p32m|SOC
Skipping incorrect answer for p32_minutos|SOC
Skipping incorrect answer for p32_horas|SOC
Skipping incorrect answer fo

Adding batches to DB: 100%|██████████| 112/112 [2:01:54<00:00, 65.30s/it]  

Total vectors: 13269
Sample peek: {'ids': ['p1_1a_1|IDE__question', 'p1_1a_1|IDE__summary', 'p1_1a_1|IDE__implications', 'p2_1a_1|IDE__question', 'p2_1a_1|IDE__summary', 'p2_1a_1|IDE__implications', 'p3_1|IDE__question', 'p3_1|IDE__summary', 'p3_1|IDE__implications', 'p3_2|IDE__question'], 'embeddings': array([[-0.01286233, -0.02477617,  0.00845953, ..., -0.00216455,
        -0.01462473, -0.00672276],
       [-0.00771893,  0.00366222,  0.03773989, ..., -0.00310992,
        -0.00857366, -0.0436573 ],
       [-0.02147629,  0.00293677,  0.03042583, ...,  0.00184429,
        -0.02441142, -0.04355532],
       ...,
       [ 0.01838034,  0.00962718,  0.02427666, ..., -0.00525712,
        -0.02070235, -0.05458009],
       [-0.00535758,  0.0059622 ,  0.01842068, ..., -0.00686241,
        -0.02355321, -0.02821548],
       [-0.00732108, -0.0118375 ,  0.03220749, ..., -0.00837447,
        -0.01164658, -0.00516163]], shape=(10, 1536)), 'documents': ['IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asoc

### filtrado de variables mal etiquetadas, etc

In [4]:
# tmp_mis_OUT_list contiene las variables que no tienen etiquetas completas - probablemente sus valores están en otra variable.... pero ese parche es para otro día. 
# TODO: parchar estas etiquetas incompletas para que no se pierdan en el futuro

# Filter keys that only contain 8.0, 9.0, 88.0, or 89.0 and not smaller numbers

def is_valid_number(x):
    try:
        # Try to convert to float, return False if it fails
        return float(str(x)) >= 8.0
    except ValueError:
        return False


# Filter keys based on value_labels containing 9.0 but no values below 8.0
filtered_keys = {}
for t_k, table_data in enc_dict.items():
    metadata = table_data.get('metadata', {})
    value_labels = metadata.get('variable_value_labels', {})
    
    matching_vars = {}
    for var_name, labels in value_labels.items():
        # Convert all numeric-like keys to float, keep non-numeric as is
        numeric_keys = []
        non_numeric_keys = []
        
        for key in labels.keys():
            try:
                numeric_keys.append(float(str(key)))
            except ValueError:
                non_numeric_keys.append(key)
        
        # Check if any key is 9.0 and no numeric keys are less than 8.0
        if numeric_keys:
            if 9.0 in numeric_keys and all(x >= 8.0 for x in numeric_keys):
                matching_vars[var_name] = labels
        elif non_numeric_keys:  # If only non-numeric keys, include it
            matching_vars[var_name] = labels
            
    if matching_vars:
        filtered_keys[t_k] = matching_vars
        # Additional filter for var_name starting with 'p' and not 's'
        filtered_keys = {
            t_k: {var_name: labels 
                  for var_name, labels in matching_vars.items() 
                  if var_name.startswith('p') and not var_name.startswith('s')
            }
            for t_k, matching_vars in filtered_keys.items()
        }

        # Remove empty dictionaries
        filtered_keys = {k: v for k, v in filtered_keys.items() if v}

tmp_mis_OUT_dict = {enc_nom_dict[ky]: list(filtered_keys[ky].keys()) for ky in filtered_keys.keys()}
tmp_mis_OUT_list = ['|'.join([st_var, st_tab]) for st_tab in tmp_mis_OUT_dict.keys() for st_var in tmp_mis_OUT_dict[st_tab]]
tmp_mis_OUT_list


['p14_1|CUL',
 'p14_2|CUL',
 'p14_3|CUL',
 'p14_4|CUL',
 'p14_5|CUL',
 'p14_6|CUL',
 'p14_7|CUL',
 'p14_8|CUL',
 'p15_1|CUL',
 'p15_2|CUL',
 'p15_3|CUL',
 'p15_4|CUL',
 'p15_5|CUL',
 'p15_6|CUL',
 'p15_7|CUL',
 'p15_8|CUL',
 'p15_9|CUL',
 'p15_10|CUL',
 'p18_1|GLO',
 'p18_2|GLO',
 'p18_3|GLO',
 'p18_4|GLO',
 'p18_5|GLO',
 'p18_6|GLO',
 'p18_7|GLO',
 'p58aa_1|GLO',
 'p3_2|GEN',
 'p4_2|GEN',
 'p57_2|GEN',
 'p58_2|GEN']

In [6]:
from tqdm import tqdm
import chromadb
from chromadb import PersistentClient                            
from chromadb.config import Settings, DEFAULT_TENANT, DEFAULT_DATABASE  
import tiktoken

# --------------------------------------------------------------------
# 1) Configure your persistent client (v0.4+)
# --------------------------------------------------------------------
DB_PERSIST_DIR = "./db_f1"
DB_NAME        = "enc_dbf1"
reset_db       = False

client = PersistentClient(
    path     = DB_PERSIST_DIR,
    settings = Settings(allow_reset=True),   # allow resetting on-disk state
    tenant   = DEFAULT_TENANT,
    database = DEFAULT_DATABASE,
)  # :contentReference[oaicite:0]{index=0}

# --------------------------------------------------------------------
# 2) Create or reset the collection
# --------------------------------------------------------------------
if reset_db:
    # deletes the whole collection directory if it exists
    try:
        client.delete_collection(name=DB_NAME)     # :contentReference[oaicite:1]{index=1}
    except ValueError:
        pass

db_f1 = client.get_or_create_collection(
    name               = DB_NAME,
    embedding_function = embedding_fun_openai
)

In [ ]:
# remover variables mal etiquetadas

facet_ids= ['__question', '__summary', '__implications']

tmp_ids_list = db_f1.get()['ids'].copy()

var_OUT_lst = [var_st + facet_st for var_st in tmp_mis_OUT_list for facet_st in facet_ids]

# variables en db_f1 que deben ser removidas
var_OUT_lst = [id for id in var_OUT_lst if id in tmp_ids_list]

db_f1.delete(ids=var_OUT_lst)
print("Total vectors:", db_f1.count())
print("Sample peek:", db_f1.peek())

Total vectors: 13191
Sample peek: {'ids': ['p1_1a_1|IDE__question', 'p1_1a_1|IDE__summary', 'p1_1a_1|IDE__implications', 'p2_1a_1|IDE__question', 'p2_1a_1|IDE__summary', 'p2_1a_1|IDE__implications', 'p3_1|IDE__question', 'p3_1|IDE__summary', 'p3_1|IDE__implications', 'p3_2|IDE__question'], 'embeddings': array([[-0.01286233, -0.02477617,  0.00845953, ..., -0.00216455,
        -0.01462473, -0.00672276],
       [-0.00771893,  0.00366222,  0.03773989, ..., -0.00310992,
        -0.00857366, -0.0436573 ],
       [-0.02147629,  0.00293677,  0.03042583, ...,  0.00184429,
        -0.02441142, -0.04355532],
       ...,
       [ 0.01838034,  0.00962718,  0.02427666, ..., -0.00525712,
        -0.02070235, -0.05458009],
       [-0.00535758,  0.0059622 ,  0.01842068, ..., -0.00686241,
        -0.02355321, -0.02821548],
       [-0.00732108, -0.0118375 ,  0.03220749, ..., -0.00837447,
        -0.01164658, -0.00516163]], shape=(10, 1536)), 'documents': ['IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asoc

['p14_1|CUL__question',
 'p14_1|CUL__summary',
 'p14_1|CUL__implications',
 'p14_2|CUL__question',
 'p14_2|CUL__summary',
 'p14_2|CUL__implications',
 'p14_3|CUL__question',
 'p14_3|CUL__summary',
 'p14_3|CUL__implications',
 'p14_4|CUL__question',
 'p14_4|CUL__summary',
 'p14_4|CUL__implications',
 'p14_5|CUL__question',
 'p14_5|CUL__summary',
 'p14_5|CUL__implications',
 'p14_6|CUL__question',
 'p14_6|CUL__summary',
 'p14_6|CUL__implications',
 'p14_7|CUL__question',
 'p14_7|CUL__summary',
 'p14_7|CUL__implications',
 'p14_8|CUL__question',
 'p14_8|CUL__summary',
 'p14_8|CUL__implications',
 'p15_1|CUL__question',
 'p15_1|CUL__summary',
 'p15_1|CUL__implications',
 'p15_2|CUL__question',
 'p15_2|CUL__summary',
 'p15_2|CUL__implications',
 'p15_3|CUL__question',
 'p15_3|CUL__summary',
 'p15_3|CUL__implications',
 'p15_4|CUL__question',
 'p15_4|CUL__summary',
 'p15_4|CUL__implications',
 'p15_5|CUL__question',
 'p15_5|CUL__summary',
 'p15_5|CUL__implications',
 'p15_6|CUL__question',
 

### tests iniciales

In [ ]:
# generación de prompt para seleccionar analizar las tablas generadas . v2. SIN query.
# generación aumentada: prompt + pregunta + documentos 

tmp_nsnc_val = '2'
tmp_topic = None


def create_prompt_sum2(table_key, tmp_markdown_tables):
    """
    Creates a prompt for analyzing a table and answering a query.

    Parameters:
        query (str): The query to be answered.
        table_key (str): The key of the table to be analyzed.
        tmp_markdown_tables (dict): A dictionary containing markdown tables.
        tmp_nsnc_val (str): The threshold percentage for 'No sabe/ No contesta'.

    Returns:
        str: The generated prompt.
    """

    tmp_tab_st = tmp_markdown_tables[table_key]
    tmp_topic = tmp_tab_st.split('|')[1].replace('_', ' ').lower().strip()

    # subtasks
    subtask_str_1 = f"""
    write a short summary of the table; note that if the 'no answer/ does not know' option is present, you should include it in the summary if it too high (higher than {tmp_nsnc_val}%) and explain what it means. Otherwise, do not mention it. 
    you will include the percentages of all the options you discuss in this summary, with format (dd.d%) where d is a digit and the number represents the percentage of respondents that selected that option.
    """

    subtask_str_2 = f"""
    you will 1) think about the table you just described and answer the question: "Why is this table important for me as an expert in TOPIC "{tmp_topic}"? You will write this reason in a short sentence.
    And then you will 2) think how other experts in TOPIC "{tmp_topic}" could use these results and you new insights about their importance, and write a short paragraph about it. 

    If at first the relationship between the table and the TOPIC "{tmp_topic}" is not clear, you should recall that these questions are part of a larger questionnaire about TOPIC "{tmp_topic}". 
    This means that respondents knew that they were answering questions about TOPIC "{tmp_topic}", and that they were asked to answer these questions in the context of TOPIC "{tmp_topic}".
    If you think the table is not relevant to the TOPIC, do NOT say so BUT make an effort to imagine how they may be related and provide an answer.
    """

    # general prompt
    prompt = f"""
    You are an expert quantitative analyst who speaks English and Spanish fluently, and can comprehend any combination of the two languages. 
    You are also an expert in TOPIC "{tmp_topic}". If you do not understand what  "{tmp_topic}" is, make sure to translate it to English for your own work. 
    You will return your answer in English, but you can use Spanish to help you understand the topic.
    You are an expert in TOPIC "{tmp_topic}" capable of discussing quantitative results like tables and qualitative topics like opinions and values about it. 

    Your task is to analyze a TABLE containing the results of an opinion survey. 
    Note that the HEADER of the TABLE contains the question, the FIRST COLUMN contains the answer options, and the SECOND COLUMN contains the percentage of people that selected that answered each option.
    This is the TABLE: {tmp_tab_st}
    
    In your analysis you will: A) {subtask_str_1}, and B) {subtask_str_2}

    You will return a python dict with the following keys: 'summary' for tast A and 'implication' for task B.
    The values of the dict should be the answers to each task.
    IMPORTANT: The dict should be formatted as follows: {{'summary': '...', 'implication': '...'}}. 
    Make sure to return only correctly formatted python dict, without any code block formatting, markdown, or additional text.

    """
    return prompt

# Example usage:
prompt = create_prompt_sum2('p46_4|IDE', mkdown_tables)


print(prompt)


    You are an expert quantitative analyst who speaks English and Spanish fluently, and can comprehend any combination of the two languages. 
    You are also an expert in TOPIC "identidad y valores". If you do not understand what  "identidad y valores" is, make sure to translate it to English for your own work. 
    You will return your answer in English, but you can use Spanish to help you understand the topic.
    You are an expert in TOPIC "identidad y valores" capable of discussing quantitative results like tables and qualitative topics like opinions and values about it. 

    Your task is to analyze a TABLE containing the results of an opinion survey. 
    Note that the HEADER of the TABLE contains the question, the FIRST COLUMN contains the answer options, and the SECOND COLUMN contains the percentage of people that selected that answered each option.
    This is the TABLE: | IDENTIDAD_Y_VALORES|Y Ahora dígame; en su opinión ¿qué tanto los mexicanos mienten para obtener un bene

In [ ]:
# llm_caller_smry() llama al modelo para crear resumenes y extrapolaciones de expertos

# 1. preparar datos

# crear promtps para cada tabla
tmp_pmt_lst= [create_prompt_sum2(ky_st, mkdown_tables) for ky_st in mkdown_tables.keys()]

# preparar listas con documentos y ids

def llm_caller_smry(ptmp_pmt_lstmt_lst = tmp_pmt_lst, tmp_pregs_dict = mkdown_tables, tmp_mkddwn_dict = mkdown_tables):
    tmp_docs= ptmp_pmt_lstmt_lst
    tmp_ids = tmp_mkddwn_dict.keys()

    docs = [ str(doc).strip() for doc in tmp_pregs_dict.values() if doc is not None and str(doc).strip() != "" ]
    ids = [ key for key, doc in tmp_pregs_dict.items() if doc is not None and str(doc).strip() != "" ]

    # batching
    tmp_bch_list = batch_documents(tmp_docs, tmp_ids, max_tokens=8192, encoding_name="cl100k_base")

    # contador e indicador de progreso
    total_rem = sum(len(batch_keys) for _, batch_keys in tmp_bch_list)

    tmp_answer_dict = {}

    # llamdaas al llm
    with tqdm(total=total_rem, desc="Making LLM calls") as pbar:
        for batch_docs, batch_keys in tmp_bch_list:
            for prompt, key in zip(batch_docs, batch_keys):
                tmp_answer_dict[key] = get_answer(prompt=prompt)

                pbar.update(1)

                # print(f'key = {key}')
                # print(f'answer = {tmp_answer_dict[key]}')
    

    return tmp_answer_dict

In [ ]:
# # exctacción/ evaluación de respuestas
# langchain and pydantic should take care of this!

# import ast

# answers_dict = {}
# for key, resp_text in tmp_answer_dict.items():
#     try:
#         # try to parse the response as a Python literal (dict)
#         answers_dict[key] = ast.literal_eval(resp_text)
#     except (ValueError, SyntaxError):
#         # if it fails, mark as incorrect
#         answers_dict[key] = "incorrect"

# # inspect the result
# answers_dict

{'p1_1a_1|IDE': 'incorrect',
 'p2_1a_1|IDE': 'incorrect',
 'p5_1a|IDE': 'incorrect',
 'p15_1a|IDE': 'incorrect',
 'p26_1a_1|IDE': {'summary': 'The table presents survey results regarding the meaning of success in life among respondents. The most common response was having material well-being (Trabajo, empleo, ganar dinero, etc.) at 31.33%, followed by achieving goals, dreams, and objectives at 13.00%. Other notable mentions include having personal autonomy and development (10.75%), and good emotional relationships (10.17%). A concerning figure is the 8.42% of respondents who answered "No sabe/ No contesta," indicating uncertainty or a lack of clarity about their definition of success. This could point to a need for deeper reflection on personal values and identity.',
  'implication': 'As an expert in identidad y valores, this table highlights critical perspectives on what individuals consider valuable in their lives, emphasizing the prevailing importance of material well-being. Additio

In [ ]:
### generación y actualización de dbf1 con todas las variables del proyecto
# tmp_var_dict ahora tendrá todas las variables de todo el proyecto

rev_enc_nom_dict = {v: k for k, v in enc_nom_dict.items()}
tmp_pregs_dict = pregs_dict.copy()
# remover Ponds que se hayan colado

tmp_pregs_dict = {ky: val for ky, val in pregs_dict.items()
    if not (ky.split('|')[0].startswith('Pond') or ky.split('|')[0].startswith('pond'))}


tmp_var_dict = {}


for k, v in tmp_pregs_dict.items():
    p_id, t_id = k.split('|')
    #print(p_id, t_id)
    df = enc_dict[rev_enc_nom_dict[t_id]]['dataframe']
    tmp_cols = df.filter(regex= 'Pond|pond').columns

    # ensure weight column is named 'Pondi2'
    if 'Pondi2' not in df.columns and 'pondi2' not in df.columns:
        raise KeyError(f"No 'Pondi2' or 'pondi2' column in dataframe for {t_id}")
    elif 'pondi2' in df.columns and 'Pondi2' not in df.columns:
        df.rename(columns={'pondi2': 'Pondi2'}, inplace=True)
    else: # fallback total si no da con el ponderador
        df['Pondi2'] = 1


    tmp_var = enc_dict[rev_enc_nom_dict[t_id]]['dataframe'][[p_id, 'Pondi2']].copy()

    tmp_var.columns = [p_id, 'ponderador']
    tmp_var['ponderador'] = tmp_var['ponderador'].astype(float)
    tmp_var['ponderador'] = tmp_var['ponderador'].fillna(0)
    tmp_var['ponderador'] = tmp_var['ponderador'] / tmp_var['ponderador'].sum()
    tmp_var.index.name = v
    tmp_var_dict[k] = tmp_var

# tmp_df_dict contiene los dataframes de cada variable

tmp_val_etq_dict = {}
tmp_df_dict = {}

for tmp_ky in tmp_var_dict.keys():
    p_id, t_id = tmp_ky.split('|')
    
    # extrae y limpia etiquetas de variables
    print(f'tmp_ky = {tmp_ky}')

    print(f'p_id = {p_id}')
    print(f't_id = {t_id}')
    tmp_etq_dict = enc_dict[rev_enc_nom_dict[t_id]]['metadata']['variable_value_labels'].copy()
    # OJO: a_1 y _a son las etiquetas sin cerrar... o no? 
    #tmp_etq_dict= {k.replace('a_1','').replace('a', ''): v for k, v in tmp_etq_dict.items()}

    if p_id not in tmp_etq_dict.keys():
        print(f'No se encuentra {p_id} en {tmp_etq_dict.keys()}')
        if p_id not in tmp_etq_dict.keys():
            print(f'No se encuentra {p_id} en {tmp_etq_dict.keys()}')
            continue

    tmp_etq = tmp_etq_dict[p_id].copy()
    tmp_etq = {k: v.replace('(esp)', '') for k, v in tmp_etq.items()}
    tmp_etq = {k: v.strip() for k, v in tmp_etq.items()}

    #limpia 8 o 98
    max_key = max(tmp_etq.keys())

    if max_key == 99.0:
        tmp_etq[max_key] = 'No sabe/ No contesta'
        tmp_etq.pop(98.0, None)
    elif max_key == 9.0:
        tmp_etq[max_key] = 'No sabe/ No contesta'
        tmp_etq.pop(8.0, None)

    tmp_val_etq_dict[tmp_ky] = tmp_etq

    # calcula y ajusta totales ponderados

    # tmp_pct_df =tmp_var_dict[tmp_ky].iloc[:, 0].value_counts(normalize=True).to_frame(name='%')
    tmp_pct_df = calculate_weighted_proportion(tmp_var_dict[tmp_ky], p_id, weight_var='ponderador', normalize=True)
    tmp_pct_df = tmp_pct_df.rename(columns={'proportion': '%'})
    # tmp_pnd_df = tmp_var_dict[tmp_ky].groupby(p_id)['ponderador'].sum()

    # tmp_pct_df = tmp_pct_df.mul(tmp_pnd_df, axis = 0)

    if max(tmp_pct_df.index) == 99.0:
        if 98.0 in tmp_pct_df.index:
            tmp_pct_df.loc[99.0] += tmp_pct_df.loc[98.0]
            tmp_pct_df = tmp_pct_df.drop(98.0)

    elif max(tmp_pct_df.index) == 9.0:
        if 8.0 in tmp_pct_df.index:
            tmp_pct_df.loc[9.0] += tmp_pct_df.loc[8.0]
            tmp_pct_df = tmp_pct_df.drop(8.0)
            
            

    # Use the values of the subdict as index
    tmp_pct_df.index = tmp_pct_df.index.map(tmp_val_etq_dict[tmp_ky])
    # print(tmp_val_etq_dict[tmp_ky])
    
    tmp_pct_df.index.name = tmp_var_dict[tmp_ky].index.name

    # print(tmp_pct_df)    


    # Store the dataframe in the dictionary
    tmp_df_dict[tmp_ky] = tmp_pct_df.mul(100).round(2)


tmp_markdown_tables = {key: dataframe_to_markdown(df) for key, df in tmp_df_dict.items()}

# vars en tmp_markdown_tables para actualizar db_f1
vars_en_mkd = set(tmp_markdown_tables.keys())

tmp_markdown_tables
# carga de db_f1 existente

from tqdm import tqdm
import chromadb
from chromadb import PersistentClient                             # CHANGED
from chromadb.config import Settings, DEFAULT_TENANT, DEFAULT_DATABASE  # CHANGED
import tiktoken

# --------------------------------------------------------------------
# 1) Configure your persistent client (v0.4+)
# --------------------------------------------------------------------
DB_PERSIST_DIR = "./db_f1"
DB_NAME        = "enc_dbf1"
reset_db       = False ## No ejecutar en True: borrará la base!!

client = PersistentClient(
    path     = DB_PERSIST_DIR,
    settings = Settings(allow_reset=True),   # allow resetting on-disk state
    tenant   = DEFAULT_TENANT,
    database = DEFAULT_DATABASE,
)  # :contentReference[oaicite:0]{index=0}
# --------------------------------------------------------------------
# 2) Create or reset the collection
# --------------------------------------------------------------------
if reset_db:
    # deletes the whole collection directory if it exists
    try:
        client.delete_collection(name=DB_NAME)     # :contentReference[oaicite:1]{index=1}
    except ValueError:
        pass

db_f1 = client.get_or_create_collection(
    name               = DB_NAME,
    embedding_function = embedding_fun_openai
)

vars_en_dbf1 = set([st.split('__')[0] for st in db_f1.get()['ids']])

db_f1.peek()
# actualización de db_f1 con vars nuevas
# identificación de vars faltantes

vars_falt = list(vars_en_mkd - vars_en_dbf1)
print(f'{len(vars_falt)} vars faltantes')

db_f1.get(ids = vars_falt)
upd_pregs_dict = {k:v for k, v in tmp_pregs_dict.items() if k in vars_falt}
upd_mkddwn_dict = {k: v for k, v in tmp_markdown_tables.items() if k in vars_falt}

In [24]:
# from pydantic import BaseModel
# from langchain.output_parsers import PydanticOutputParser
# import pickle

# # 1. Define single-entry Pydantic model
# class SummaryEntry(BaseModel):
#     summary: str
#     implication: str

# # 2. Create parser and get JSON format instructions
# parser_summary = PydanticOutputParser(pydantic_object=SummaryEntry)
# format_instructions = parser_summary.get_format_instructions()

# # 3. Helper to call LLM and parse response to dict

# def get_structured_summary(
#     table_key: str,
#     model_name: str = 'gpt-4o-mini-2024-07-18',
#     temperature: float = 0.9
# ) -> dict:
#     prompt = create_prompt_sum2(
#         table_key,
#         mkdown_tables,
#         nsnc_threshold=2.0,
#         topic=None,
#         format_instructions=format_instructions
#     )
#     content = get_answer(prompt, model=model_name, temperature=temperature)
#     entry = parser_summary.parse(content)
#     return entry.model_dump()

# # 4. Wrap original LLM caller to use structured parser and save partial results
# original_keys = list(mkdown_tables.keys())[:10]
# structured_results = {}

# for idx, key in enumerate(original_keys, 1):
#     try:
#         structured_results[key] = get_structured_summary(key)
#     except Exception as e:
#         structured_results[key] = {'error': str(e)}
#     # Save checkpoint every 5 tables
#     if idx % 5 == 0 or idx == len(original_keys):
#         with open('structured_summary_checkpoint.pkl', 'wb') as f:
#             pickle.dump(structured_results, f)
#         print(f"Checkpoint saved after {idx} tables.")

Checkpoint saved after 5 tables.
Checkpoint saved after 10 tables.
